# 🎯 Test spotter sobre UN video broadcast

¿El spotter pre-entrenado encuentra el gol/tiro cuando el input es **broadcast real** (con replays/gráficos),
en vez de la grabación de pantalla? Corre los 2 modelos a umbral bajo (0.1) sobre un solo video.

> Activá **GPU** (Runtime → Change runtime type → GPU / L4).


In [ ]:
import torch
print('GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'SIN GPU — activá una')

In [ ]:
# Clonar T-DEED y aplicar los fixes locales (imports opcionales, device flexible, num_workers)
import os
BRANCH = 'events-model'
REPO = '/content/T-DEED'
if not os.path.exists(REPO):
    !git clone -q https://github.com/arturxe2/T-DEED.git {REPO}
if not os.path.exists('/content/ncf'):
    !git clone -q -b {BRANCH} https://github.com/pipachiesa/ncf_event_tracker.git /content/ncf
%cd {REPO}
!git apply --check /content/ncf/events_model/tdeed/tdeed_local_fixes.patch && git apply /content/ncf/events_model/tdeed/tdeed_local_fixes.patch && echo 'patch aplicado'
!cp /content/ncf/events_model/tdeed/run_spotter.py .
!pip install -q timm tabulate gdown

In [ ]:
# Checkpoints públicos de T-DEED (carpeta completa; usamos SoccerNet y SoccerNetBall)
import os, glob, shutil
!gdown -q --folder https://drive.google.com/drive/folders/1sxZalU_hCwL8ITZCU9VqSWE8dB94lJty -O /content/tdeed_ckpts || echo 'ojo: revisar descarga'
os.makedirs('checkpoints', exist_ok=True)
for d in glob.glob('/content/tdeed_ckpts/*'):
    dst = os.path.join('checkpoints', os.path.basename(d))
    if os.path.isdir(d) and not os.path.exists(dst): shutil.move(d, dst)
print('checkpoints:', sorted(os.listdir('checkpoints')))

In [ ]:
from google.colab import files
import os
print('Subí tu video broadcast (mp4) ...')
a = files.upload()
VIDEO = os.path.abspath(list(a.keys())[0])
print('video:', VIDEO)


In [ ]:
# Los 2 modelos a umbral bajo (0.1) para maximizar chance de agarrar el gol
MODELS = ['SoccerNet_small', 'SoccerNetBall_challenge1']   # 17-clases (Goal/Shots/Foul) y ball-action (Shot/Goal)
for model in MODELS:
    print(f"\n{'='*70}\n>>> {model} | thr=0.1\n{'='*70}")
    !python3 run_spotter.py --model {model} --video "{VIDEO}" --threshold 0.1


In [ ]:
# Resumen: releer todos los JSON y mostrar solo Goal/Shot/Foul lado a lado
import json, glob, os
KEY = ('goal', 'shot', 'foul')
for path in sorted(glob.glob('spotter_results/*.json')):
    r = json.load(open(path))
    evs = [p for p in r['predictions'] if any(k in p['label'].lower() for k in KEY)]
    evs.sort(key=lambda p: p['frame'])
    print(f"\n--- {os.path.basename(path)} ({len(evs)} eventos clave) ---")
    for p in evs:
        t = p['frame'] / r['fps']
        print(f"  {int(t//60):02d}:{t%60:04.1f}  {p['label']:<20} conf={p['confidence']:.3f}")